In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_csv('flipkart_com-ecommerce_sample.csv')

In [ ]:
print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nNull counts:\n", df.isnull().sum())
print("\nFirst 3 rows:\n", df.head(3))

Shape: (20000, 15)

Columns:
 ['uniq_id', 'crawl_timestamp', 'product_url', 'product_name', 'product_category_tree', 'pid', 'retail_price', 'discounted_price', 'image', 'is_FK_Advantage_product', 'description', 'product_rating', 'overall_rating', 'brand', 'product_specifications']

Null counts:
 uniq_id                       0
crawl_timestamp               0
product_url                   0
product_name                  0
product_category_tree         0
pid                           0
retail_price                 78
discounted_price             78
image                         3
is_FK_Advantage_product       0
description                   2
product_rating                0
overall_rating                0
brand                      5864
product_specifications       14
dtype: int64

First 3 rows:
                             uniq_id            crawl_timestamp  \
0  c2d766ca982eca8304150849735ffef9  2016-03-25 22:59:23 +0000   
1  7f7036a6d550aaa89d34c77bd39a5e48  2016-03-25 22:59:23 +0000

In [ ]:
# ── STEP 1: Drop rows missing critical fields ──────────────────────────
df_clean = df.dropna(subset=['product_name', 'retail_price', 'discounted_price'])
print(f"Rows after dropping nulls: {len(df_clean)}")

# ── STEP 2: Fix price columns (already numeric, just ensure float) ─────
df_clean['retail_price'] = pd.to_numeric(df_clean['retail_price'], errors='coerce')
df_clean['discounted_price'] = pd.to_numeric(df_clean['discounted_price'], errors='coerce')

# ── STEP 3: Calculate discount percentage ─────────────────────────────
df_clean['discount_pct'] = (
    (df_clean['retail_price'] - df_clean['discounted_price'])
    / df_clean['retail_price'] * 100
).round(1)

# ── STEP 4: Extract main category from product_category_tree ──────────
df_clean['main_category'] = (
    df_clean['product_category_tree']
    .str.replace('["', '', regex=False)
    .str.split('>>')
    .str[0]
    .str.strip()
)

# ── STEP 5: Fix overall_rating ─────────────────────────────────────────
# Replace "No rating available" with NaN, then convert to numeric
df_clean['overall_rating'] = df_clean['overall_rating'].replace(
    'No rating available', np.nan
)
df_clean['overall_rating'] = pd.to_numeric(df_clean['overall_rating'], errors='coerce')

# ── STEP 6: Fix product_rating same way ───────────────────────────────
df_clean['product_rating'] = df_clean['product_rating'].replace(
    'No rating available', np.nan
)
df_clean['product_rating'] = pd.to_numeric(df_clean['product_rating'], errors='coerce')

# ── STEP 7: Fill missing brand with "Unknown" ─────────────────────────
df_clean['brand'] = df_clean['brand'].fillna('Unknown')

# ── STEP 8: Add price band column ─────────────────────────────────────
df_clean['price_band'] = pd.cut(
    df_clean['discounted_price'],
    bins=[0, 500, 2000, 10000, 999999],
    labels=['Budget', 'Mid', 'Premium', 'Luxury']
)

# ── STEP 9: Keep only useful columns ──────────────────────────────────
df_clean = df_clean[[
    'product_name', 'main_category', 'brand',
    'retail_price', 'discounted_price', 'discount_pct',
    'overall_rating', 'product_rating', 'price_band',
    'is_FK_Advantage_product'
]]

# ── STEP 10: Final check ───────────────────────────────────────────────
print("\nCleaned shape:", df_clean.shape)
print("\nNull counts after cleaning:\n", df_clean.isnull().sum())
print("\nSample categories:", df_clean['main_category'].value_counts().head(10))
print("\nPrice band distribution:\n", df_clean['price_band'].value_counts())
print("\nDiscount % stats:\n", df_clean['discount_pct'].describe().round(1))

Rows after dropping nulls: 19922

Cleaned shape: (19922, 10)

Null counts after cleaning:
 product_name                   0
main_category                  0
brand                          0
retail_price                   0
discounted_price               0
discount_pct                   0
overall_rating             18083
product_rating             18083
price_band                     0
is_FK_Advantage_product        0
dtype: int64

Sample categories: main_category
Clothing                      6171
Jewellery                     3522
Footwear                      1225
Mobiles & Accessories         1097
Automotive                    1010
Home Decor & Festive Needs     927
Beauty and Personal Care       709
Home Furnishing                700
Kitchen & Dining               645
Computers                      573
Name: count, dtype: int64

Price band distribution:
 price_band
Budget     9444
Mid        8592
Premium    1061
Luxury      825
Name: count, dtype: int64

Discount % stats:
 count   

/tmp/ipykernel_704/731568364.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['retail_price'] = pd.to_numeric(df_clean['retail_price'], errors='coerce')
/tmp/ipykernel_704/731568364.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['discounted_price'] = pd.to_numeric(df_clean['discounted_price'], errors='coerce')
/tmp/ipykernel_704/731568364.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = valu

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Fix the SettingWithCopyWarning properly
df_clean = df.dropna(subset=['product_name', 'retail_price', 'discounted_price']).copy()

df_clean['retail_price'] = pd.to_numeric(df_clean['retail_price'], errors='coerce')
df_clean['discounted_price'] = pd.to_numeric(df_clean['discounted_price'], errors='coerce')
df_clean['discount_pct'] = ((df_clean['retail_price'] - df_clean['discounted_price']) / df_clean['retail_price'] * 100).round(1)
df_clean['main_category'] = df_clean['product_category_tree'].str.replace('["', '', regex=False).str.split('>>').str[0].str.strip()
df_clean['overall_rating'] = pd.to_numeric(df_clean['overall_rating'].replace('No rating available', np.nan), errors='coerce')
df_clean['product_rating'] = pd.to_numeric(df_clean['product_rating'].replace('No rating available', np.nan), errors='coerce')
df_clean['brand'] = df_clean['brand'].fillna('Unknown')
df_clean['price_band'] = pd.cut(df_clean['discounted_price'], bins=[0, 500, 2000, 10000, 999999], labels=['Budget', 'Mid', 'Premium', 'Luxury'])

df_clean = df_clean[[
    'product_name', 'main_category', 'brand',
    'retail_price', 'discounted_price', 'discount_pct',
    'overall_rating', 'product_rating', 'price_band',
    'is_FK_Advantage_product'
]]

# Save cleaned CSV to Drive
df_clean.to_csv('/content/drive/MyDrive/flipkart_cleaned.csv', index=False)
print("Saved! Rows:", len(df_clean))
print("No warnings this time.")

Saved! Rows: 19922
No warnings this time.


In [ ]:
# Authenticate with your GCP account
from google.colab import auth
auth.authenticate_user()
print("Authenticated!")

Authenticated!


In [ ]:
from google.cloud import bigquery, storage

PROJECT_ID = 'gen-lang-client-0891751397'
BUCKET_NAME = 'flipkart-dashboard-bucket'
DATASET_NAME = 'flipkart_dataset'
TABLE_NAME = 'products'

# ── Create GCS bucket ──────────────────────────────────────────────────
storage_client = storage.Client(project=PROJECT_ID)

try:
    bucket = storage_client.create_bucket(BUCKET_NAME, location='US')
    print(f"Bucket created: {BUCKET_NAME}")
except Exception as e:
    bucket = storage_client.bucket(BUCKET_NAME)
    print(f"Bucket already exists — continuing")

# ── Upload CSV to GCS ──────────────────────────────────────────────────
blob = bucket.blob('flipkart/flipkart_cleaned.csv')
blob.upload_from_filename('/content/drive/MyDrive/flipkart_cleaned.csv')
print("CSV uploaded to GCS!")

# ── Create BigQuery dataset ────────────────────────────────────────────
bq_client = bigquery.Client(project=PROJECT_ID)

try:
    dataset = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_NAME}")
    dataset.location = 'US'
    bq_client.create_dataset(dataset)
    print(f"Dataset created: {DATASET_NAME}")
except Exception as e:
    print(f"Dataset already exists — continuing")

# ── Load CSV into BigQuery ─────────────────────────────────────────────
uri = f'gs://{BUCKET_NAME}/flipkart/flipkart_cleaned.csv'
table_ref = f"{PROJECT_ID}.{DATASET_NAME}.{TABLE_NAME}"

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

load_job = bq_client.load_table_from_uri(uri, table_ref, job_config=job_config)
load_job.result()

table = bq_client.get_table(table_ref)
print(f"\nSuccess! {table.num_rows} rows loaded into BigQuery.")
print(f"Table: {table_ref}")

Bucket created: flipkart-dashboard-bucket
CSV uploaded to GCS!
Dataset created: flipkart_dataset

Success! 19922 rows loaded into BigQuery.
Table: gen-lang-client-0891751397.flipkart_dataset.products


In [1]:
# Download notebook to your computer
from google.colab import files

# Save SQL views to a file first
sql_views = """
-- View 1: Category performance
CREATE OR REPLACE VIEW flipkart_dataset.v_category_performance AS
SELECT main_category, COUNT(*) AS total_products,
  ROUND(AVG(retail_price), 0) AS avg_retail_price,
  ROUND(AVG(discounted_price), 0) AS avg_discounted_price,
  ROUND(AVG(discount_pct), 1) AS avg_discount_pct
FROM flipkart_dataset.products
WHERE main_category IS NOT NULL
GROUP BY main_category ORDER BY total_products DESC;

-- View 2: Price band distribution
CREATE OR REPLACE VIEW flipkart_dataset.v_price_bands AS
SELECT main_category, price_band, COUNT(*) AS product_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY main_category), 1) AS pct_in_category
FROM flipkart_dataset.products
GROUP BY main_category, price_band;

-- View 3: Discount tier analysis
CREATE OR REPLACE VIEW flipkart_dataset.v_discount_tiers AS
SELECT main_category,
  CASE WHEN discount_pct < 10 THEN '1. Low (under 10%)'
       WHEN discount_pct BETWEEN 10 AND 30 THEN '2. Medium (10-30%)'
       WHEN discount_pct BETWEEN 30 AND 50 THEN '3. High (30-50%)'
       ELSE '4. Very High (above 50%)' END AS discount_tier,
  COUNT(*) AS product_count
FROM flipkart_dataset.products
WHERE discount_pct IS NOT NULL AND main_category IS NOT NULL
GROUP BY main_category, discount_tier ORDER BY main_category, discount_tier;

-- View 4: Top 10 competitive categories
CREATE OR REPLACE VIEW flipkart_dataset.v_competitive_categories AS
SELECT main_category, COUNT(*) AS listing_count,
  ROUND(AVG(discount_pct), 1) AS avg_discount_pct,
  ROUND(MAX(discount_pct) - MIN(discount_pct), 1) AS discount_range,
  ROUND(MAX(discounted_price), 0) AS max_price,
  ROUND(MIN(discounted_price), 0) AS min_price
FROM flipkart_dataset.products
WHERE main_category IS NOT NULL
GROUP BY main_category ORDER BY listing_count DESC LIMIT 10;

-- View 5: FK Advantage analysis
CREATE OR REPLACE VIEW flipkart_dataset.v_fk_advantage AS
SELECT main_category, is_FK_Advantage_product,
  COUNT(*) AS product_count,
  ROUND(AVG(discount_pct), 1) AS avg_discount_pct,
  ROUND(AVG(discounted_price), 0) AS avg_price
FROM flipkart_dataset.products
WHERE main_category IS NOT NULL
GROUP BY main_category, is_FK_Advantage_product;
"""

with open('sql_views.sql', 'w') as f:
    f.write(sql_views)

files.download('sql_views.sql')
print("Downloaded sql_views.sql")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded sql_views.sql
